In [1]:
from pathlib import Path
import csv


folder = (
    Path.home() / "Downloads"
    / "gdc-client_2.3_Windows_x64-py3.8-windows-2019"
    / "gdc-client_2.3_Windows_x64"
)

manifest = folder / "gdc_manifest.2026-10-11.163832.txt"


with open(manifest, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter="\t")
    columns = reader.fieldnames
    files = list(reader)


missing = []

for file in files:
    path = folder / file["id"] / file["filename"]

    if not path.is_file() or path.stat().st_size != int(file["size"]):
        missing.append(file)


output = folder / "missing_manifest.txt"

with open(output, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=columns, delimiter="\t")
    writer.writeheader()
    writer.writerows(missing)

print("Total files:", len(files))
print("Downloaded files:", len(files) - len(missing))
print("Missing files:", len(missing))
print("Missing manifest saved!")



Total files: 434
Downloaded files: 434
Missing files: 0
Missing manifest saved!


In [2]:
import pandas as pd
from pathlib import Path


files = list(folder.rglob("*.maf.gz"))


downloads = Path.home() / "Downloads"
sheets = list(downloads.rglob("gdc_sample_sheet.2026-10-11.tsv"))

mut_file = next(p for p in sheets if p.parent.name == "mutation")
rna_file = next(p for p in sheets if p.parent.name in ["data_rna", "database"])

mut_ss = pd.read_csv(mut_file, sep="\t")
rna_ss = pd.read_csv(rna_file, sep="\t")

patients = (
    set(mut_ss["Sample ID"].str[:12]) &
    set(rna_ss["Sample ID"].str[:12])
)


keep = [
    "Missense_Mutation", "Nonsense_Mutation",
    "Frame_Shift_Del", "Frame_Shift_Ins",
    "In_Frame_Del", "In_Frame_Ins",
    "Splice_Site", "Translation_Start_Site",
    "Nonstop_Mutation"
]

cad_genes = {

    "CDH1", "CDH2", "CDH3", "CDH11",

    "CDH13", "CDH17",

    "CDHR2", "CDHR5",

    "DSC2", "DSG2",

    "FAT1",

    "PCDH1", "PCDH17", "PCDH18",

    "PCDHGA10",

    "CLSTN1"

}

tp53 = set()
cad = set()


for f in files:
    for df in pd.read_csv(
        f, sep="\t", comment="#",
        usecols=["Hugo_Symbol", "Variant_Classification",
                 "Tumor_Sample_Barcode"],
        chunksize=50000
    ):
        df = df[df["Variant_Classification"].isin(keep)]
        ids = df["Tumor_Sample_Barcode"].str[:12]

        tp53.update(ids[df["Hugo_Symbol"] == "TP53"])
        cad.update(ids[df["Hugo_Symbol"].isin(cad_genes)])



tp53_only = patients & (tp53 - cad)
cad_only = patients & (cad - tp53)
both = patients & tp53 & cad
neither = patients - (tp53 | cad)

print("Total MAF files:", len(files))
print("Total patients:", len(patients))
print("TP53 only:", len(tp53_only))
print("Cadherin only:", len(cad_only))
print("TP53 + Cadherin:", len(both))
print("Neither:", len(neither))
print("Total TP53:", len(tp53& patients))
print("TP53 only:", len(tp53_only))
print("TP53 + cadherin:", len(both))


Total MAF files: 434
Total patients: 404
TP53 only: 107
Cadherin only: 88
TP53 + Cadherin: 76
Neither: 133
Total TP53: 183
TP53 only: 107
TP53 + cadherin: 76


In [3]:
import pandas as pd

candidate_genes = ["CDH6", "CDH10", "FAT4"]

cols = ["Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode"]

hits = pd.concat([chunk.loc[chunk["Hugo_Symbol"].isin(candidate_genes) & chunk["Variant_Classification"].isin(keep), cols] for f in files for chunk in pd.read_csv(f, sep="\t", comment="#", usecols=cols, chunksize=50000)], ignore_index=True)

hits["patient"] = hits["Tumor_Sample_Barcode"].astype(str).str[:12]

hits = hits[hits["patient"].isin(patients)]

gene_patients = {g: set(hits.loc[hits["Hugo_Symbol"] == g, "patient"]) for g in candidate_genes}

results = pd.DataFrame({

    "Gene": candidate_genes,

    "Neither": [len(gene_patients[g] & neither) for g in candidate_genes],

    "TP53 Only": [len(gene_patients[g] & tp53_only) for g in candidate_genes]

})

print("Impact of each gene:")

print(results.to_string(index=False))

extra_patients = set(hits["patient"])

new_cad = cad | extra_patients

new_tp53_only = patients & (tp53 - new_cad)

new_cad_only = patients & (new_cad - tp53)

new_both = patients & tp53 & new_cad

new_neither = patients - (tp53 | new_cad)

print("\nNew Groups:")

print("TP53 Only:", len(new_tp53_only))

print("Cadherin Only:", len(new_cad_only))

print("TP53 + Cadherin:", len(new_both))

print("Neither:", len(new_neither))

print("Total:", len(patients))

Impact of each gene:
 Gene  Neither  TP53 Only
 CDH6        3          4
CDH10        6          6
 FAT4       21         18

New Groups:
TP53 Only: 83
Cadherin Only: 114
TP53 + Cadherin: 100
Neither: 107
Total: 404


In [4]:
cad_without_fat4 = cad


cad_with_fat4 = cad | gene_patients["FAT4"]


tp53_only_without = patients & (tp53 - cad_without_fat4)
cad_only_without = patients & (cad_without_fat4 - tp53)
both_without = patients & tp53 & cad_without_fat4
neither_without = patients - (tp53 | cad_without_fat4)


tp53_only_with = patients & (tp53 - cad_with_fat4)
cad_only_with = patients & (cad_with_fat4 - tp53)
both_with = patients & tp53 & cad_with_fat4
neither_with = patients - (tp53 | cad_with_fat4)


comparison = pd.DataFrame({
"Group": [
"TP53 Only",
"Cadherin Only",
"TP53 + Cadherin",
"Neither"
],
"Without FAT4": [
len(tp53_only_without),
len(cad_only_without),
len(both_without),
len(neither_without)
],
"With FAT4": [
len(tp53_only_with),
len(cad_only_with),
len(both_with),
len(neither_with)
]
})


comparison.loc[len(comparison)] = [
"Total",
len(patients),
len(patients)
]


assert len(tp53_only_without) + len(both_without) == 183
assert len(tp53_only_with) + len(both_with) == 183


print("Comparison of Cadherin Gene Panels:")
print(comparison.to_string(index=False))

comparison.to_csv(
"TCGA_STAD_FAT4_Comparison.csv",
index=False
)

print("\nComparison saved successfully!")

Comparison of Cadherin Gene Panels:
          Group  Without FAT4  With FAT4
      TP53 Only           107         89
  Cadherin Only            88        109
TP53 + Cadherin            76         94
        Neither           133        112
          Total           404        404

Comparison saved successfully!
